# EV User Adaptation Analysis — Ordinal Regression & Statistical Study
**Survey:** User Experience in EV Adaptation (Germany & International, Jan 2024)

**Methods:** EDA · Ordinal Regression · PCA · K-Means Clustering · Hierarchical Clustering · Mann-Whitney U Tests

> ⚠️ **To run this notebook** you must supply your own `UserPerception.xlsx` file.
> See `README.md → Data Availability` for the expected column format.
> The dataset is excluded from this repository to protect respondent privacy (GDPR).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel
from tabulate import tabulate
import math
%matplotlib inline

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECURE DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
# The raw dataset (UserPerception.xlsx) is NOT included in this repository
# to protect respondent privacy and comply with GDPR.
#
# ── Option A: Google Colab ────────────────────────────────────────────────────
#    Upload UserPerception.xlsx to your own Google Drive and authenticate below.
#    This notebook never accesses anyone else's Drive.
#
# ── Option B: Local Jupyter ───────────────────────────────────────────────────
#    Place UserPerception.xlsx in a folder called data/ next to this notebook.
# ─────────────────────────────────────────────────────────────────────────────

DATA_FILE  = 'UserPerception.xlsx'          # change filename here if yours differs
COLAB_PATH = f'/content/drive/MyDrive/{DATA_FILE}'   # adjust subfolder if needed
LOCAL_PATH = os.path.join('data', DATA_FILE)

file_path = None

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(COLAB_PATH):
        file_path = COLAB_PATH
        print(f'Running in Colab  —  file found at: {COLAB_PATH}')
    else:
        print(f'WARNING: File not found at {COLAB_PATH}')
        print('Please upload UserPerception.xlsx to your Google Drive')
        print('and update COLAB_PATH above to match its location.')
except ImportError:
    if os.path.exists(LOCAL_PATH):
        file_path = LOCAL_PATH
        print(f'Running locally  —  file found at: {LOCAL_PATH}')
    else:
        print(f'WARNING: File not found at {LOCAL_PATH}')

if file_path is None:
    raise FileNotFoundError(
        '\n\nData file could not be located.\n'
        'See README.md → Data Availability section for the expected file format.\n'
        'Contact the repository owner to request an anonymised sample dataset.'
    )

In [ ]:
df = pd.read_excel(file_path)
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
df.info()
print()
df.describe()

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
df_corr = numeric_df.corr()
print(df_corr)

## Exploratory Data Analysis (EDA)
Count plots showing the Likert-scale response distribution for each perception dimension.
Scale: **1 = Not Important → 5 = Very Important**

In [ ]:
factors = {
    'V_P': 'Vehicle Purchase Intent',
    'R_A': 'Range Anxiety',
    'C_S': 'Charging Infrastructure',
    'B_I': 'Battery Issues',
    'C_T': 'Charging Time',
    'A_D': 'Autonomous Driving',
    'V_C': 'V2X Connectivity',
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for idx, (col, label) in enumerate(factors.items()):
    sns.countplot(x=col, data=df, ax=axes[idx], palette='Blues_d')
    axes[idx].set_title(label, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Importance Rating (1–5)', fontsize=9)
    axes[idx].set_ylabel('Count', fontsize=9)

axes[-1].set_visible(False)  # hide the empty 8th subplot
plt.suptitle('Response Distribution — All EV Perception Factors',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Ordinal Regression Models
Testing whether perception factors (R_A, C_S, B_I, C_T, A_D, V_C) predict purchase intent (V_P).

Two distributions are tested:
- **Probit model** — assumes normally distributed latent variable
- **Logit model** — assumes logistically distributed latent variable

**Key finding:** R_A (Range Anxiety), A_D (Autonomous Driving), and V_C (V2X Connectivity)
show significant coefficients in both models.

In [ ]:
# ── Probit model ─────────────────────────────────────────────────────────────
mod_probit = OrderedModel(df['V_P'],
                          df[['R_A', 'C_S', 'B_I', 'C_T', 'A_D', 'V_C']],
                          distr='probit')
res_probit = mod_probit.fit(method='bfgs', disp=False)
print('=== PROBIT MODEL ===')
print(res_probit.summary())

# ── Logit model ───────────────────────────────────────────────────────────────
mod_logit = OrderedModel(df['V_P'],
                         df[['R_A', 'C_S', 'B_I', 'C_T', 'A_D', 'V_C']],
                         distr='logit')
res_logit = mod_logit.fit(method='bfgs', disp=False)
print('\n=== LOGIT MODEL ===')
print(res_logit.summary())

## Mann-Whitney U Tests — Pairwise Comparison of Perception Dimensions

**H₀:** The two perception dimensions have the same distribution across respondents
**H₁:** The two perception dimensions have different distributions

The Mann-Whitney U test is used because Likert data is **ordinal**
(not continuous), so parametric tests (t-test) are not appropriate.

**Significance threshold:** p < 0.05

In [ ]:
pairs = [
    ('R_A', 'A_D'), ('R_A', 'V_C'), ('A_D', 'V_C'),
    ('R_A', 'C_S'), ('R_A', 'B_I'), ('R_A', 'C_T'),
    ('A_D', 'C_S'), ('A_D', 'B_I'), ('A_D', 'C_T'),
    ('V_C', 'C_S'), ('V_C', 'B_I'), ('V_C', 'C_T'),
    ('V_C', 'R_A'), ('V_C', 'A_D'), ('A_D', 'R_A'),
    ('V_P', 'R_A'), ('V_P', 'A_D'), ('V_P', 'V_C'),
    ('V_P', 'C_S'), ('V_P', 'B_I'), ('V_P', 'C_T'),
    ('C_S', 'C_T'), ('C_S', 'B_I'),
]

results = []
for f1, f2 in pairs:
    stat, p = stats.mannwhitneyu(df[f1], df[f2], alternative='two-sided')
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    results.append([f'{f1} & {f2}', f'{p:.3e}', sig])

print(tabulate(results,
               headers=['Factor Pair', 'P-Value', 'Significance'],
               tablefmt='github'))
print()
print('*** p<0.001  ** p<0.01  * p<0.05  n.s. = not significant')

## Dimensionality Reduction & Clustering

### Principal Component Analysis (PCA)
Reduces 6 Likert dimensions to 2 principal components for visualisation.

### K-Means Clustering (k=3)
Segments respondents into 3 behavioural groups based on their perception profiles.

### Hierarchical Clustering (Ward Linkage)
Builds a dendrogram to show how respondents naturally group by similarity.

In [ ]:
FACTOR_COLS = ['R_A', 'C_S', 'B_I', 'C_T', 'A_D', 'V_C']
X = df[FACTOR_COLS].values

# ── PCA ───────────────────────────────────────────────────────────────────────
pca      = PCA(n_components=2)
X_pca    = pca.fit_transform(X)
var1, var2 = pca.explained_variance_ratio_ * 100
print(f'PCA: PC1 explains {var1:.1f}% | PC2 explains {var2:.1f}% | Total: {var1+var2:.1f}%')

# ── K-Means ───────────────────────────────────────────────────────────────────
kmeans   = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)
print(f'\nK-Means cluster sizes: {dict(zip(*np.unique(clusters, return_counts=True)))}')

# ── Hierarchical clustering ───────────────────────────────────────────────────
linked = linkage(X, method='ward')
print('\nHierarchical clustering complete — see dendrogram plot below')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: K-Means on PCA axes
colors  = {0: '#1a6a8a', 1: '#e67e22', 2: '#27ae60'}
markers = {0: 'o', 1: 's', 2: '^'}
names   = {0: 'Cluster A — EV Enthusiasts', 1: 'Cluster B — Cautious', 2: 'Cluster C — Low Engagement'}
for cid in [0, 1, 2]:
    mask = clusters == cid
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[cid], marker=markers[cid],
                    s=55, alpha=0.75, label=names[cid])
centres = pca.transform(kmeans.cluster_centers_)
for i, (cx, cy) in enumerate(centres):
    axes[0].scatter(cx, cy, c=colors[i], s=200, marker='*',
                    edgecolors='black', linewidth=1, zorder=5)
axes[0].set_xlabel(f'PC1 ({var1:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({var2:.1f}% variance)')
axes[0].set_title('K-Means Clusters (k=3) on PCA Space\nStars = cluster centroids',
                  fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Plot 2: Dendrogram
dendrogram(linked, ax=axes[1], color_threshold=8,
           above_threshold_color='#aaaaaa',
           no_labels=True,
           link_color_func=lambda k: '#1a6a8a')
axes[1].axhline(y=8, color='red', linewidth=1.5, linestyle='--',
                label='Cut-off → 3 clusters')
axes[1].set_title('Hierarchical Clustering Dendrogram\nWard Linkage — each leaf = 1 respondent',
                  fontweight='bold')
axes[1].set_ylabel('Ward Linkage Distance')
axes[1].set_xlabel('Respondents')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Consumer Segmentation — PCA + Clustering on 6 EV Perception Factors',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Publication-Quality Visualisations
Run this cell to save all 5 annotated plots as PNG files (for LinkedIn / GitHub / CV).
In Google Colab: download from the **Files panel** (folder icon, left sidebar) → `plots/` folder.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PUBLICATION-QUALITY VISUALISATIONS — FOR LINKEDIN / GITHUB / CV
# Saves 5 annotated PNG files to the plots/ folder
# ═══════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = '/content/plots' if os.path.exists('/content') else 'plots'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PALETTE  = ['#e8f4f8', '#90cce0', '#3a9abf', '#1a6a8a', '#0d3d52']
ACCENT   = '#1a6a8a'
BG       = '#f8f9fa'
GRID     = '#e0e0e0'
SOURCE   = 'Survey: User Experience in EV Adaptation  (Germany & International, Jan 2024, n=130)'
FACTOR_LABELS = {
    'R_A': 'Range Anxiety', 'C_S': 'Charging Infrastructure',
    'B_I': 'Battery Issues', 'C_T': 'Charging Time',
    'A_D': 'Autonomous Driving', 'V_C': 'V2X Connectivity'
}
factors = list(FACTOR_LABELS.keys())
labels  = list(FACTOR_LABELS.values())

# ── Plot 1: Grouped bar — all factors ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
x = np.arange(len(factors)); width = 0.13
likert_labels = ['1 — Not important','2','3 — Neutral','4','5 — Very important']
for i, score in enumerate([1,2,3,4,5]):
    counts = [((df[f]==score).sum()/len(df))*100 for f in factors]
    bars = ax.bar(x+i*width-2*width, counts, width, color=PALETTE[i],
                  label=likert_labels[i], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, counts):
        if val > 6:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f'{val:.0f}%', ha='center', va='bottom', fontsize=7.5, color='#333')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10, fontweight='500')
ax.set_ylabel('% of Respondents', fontsize=11); ax.set_ylim(0, 52)
ax.yaxis.grid(True, color=GRID, linewidth=0.7); ax.set_axisbelow(True)
for s in ax.spines.values(): s.set_visible(False)
ax.legend(title='Importance Rating', fontsize=8.5, loc='upper right', framealpha=0.9)
ax.set_title('How Important Are These EV Factors?\nResponse Distribution Across All Perception Dimensions',
             fontsize=13, fontweight='bold', pad=14)
ax.text(0.5,-0.14, SOURCE+'\nScale: 1 = Not Important → 5 = Very Important',
        transform=ax.transAxes, ha='center', fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0,0.06,1,1])
p1 = os.path.join(OUTPUT_DIR,'plot1_all_factors_distribution.png')
plt.savefig(p1, dpi=150, bbox_inches='tight', facecolor=BG); plt.show()
print(f'✅  Saved: {p1}')

# ── Plot 2: Diverging Likert ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
neg2=[((df[f]==1).sum()/len(df))*100 for f in factors]
neg1=[((df[f]==2).sum()/len(df))*100 for f in factors]
neu =[((df[f]==3).sum()/len(df))*100 for f in factors]
pos1=[((df[f]==4).sum()/len(df))*100 for f in factors]
pos2=[((df[f]==5).sum()/len(df))*100 for f in factors]
y=np.arange(len(factors))
base=[-n2-n1-ne/2 for n2,n1,ne in zip(neg2,neg1,neu)]
l0=base
ax.barh(y,neg2,left=l0,color='#c0392b',label='1 — Not important',height=0.55,edgecolor='white',linewidth=0.5)
l1=[l+v for l,v in zip(l0,neg2)]
ax.barh(y,neg1,left=l1,color='#e8a09a',label='2',height=0.55,edgecolor='white',linewidth=0.5)
l2=[l+v for l,v in zip(l1,neg1)]
ax.barh(y,neu, left=l2,color='#d4d4d4',label='3 — Neutral',height=0.55,edgecolor='white',linewidth=0.5)
l3=[l+v for l,v in zip(l2,neu)]
ax.barh(y,pos1,left=l3,color='#7ab8d4',label='4',height=0.55,edgecolor='white',linewidth=0.5)
l4=[l+v for l,v in zip(l3,pos1)]
ax.barh(y,pos2,left=l4,color=ACCENT,label='5 — Very important',height=0.55,edgecolor='white',linewidth=0.5)
ax.axvline(0,color='#555',linewidth=1.0,linestyle='--',alpha=0.6)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=10.5)
ax.set_xlabel('← Less Important                       More Important →', fontsize=10, color='#555')
ax.xaxis.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
for s in ax.spines.values(): s.set_visible(False)
ax.legend(loc='lower right', fontsize=8.5, title='Rating', framealpha=0.9)
ax.set_title('Diverging Likert Chart — Consumer Stance on Each EV Factor\n'
             'Bars extend left (low importance) or right (high importance) from neutral centre',
             fontsize=12, fontweight='bold', pad=12)
ax.text(0.5,-0.13, SOURCE, transform=ax.transAxes, ha='center', fontsize=8, color='#666', style='italic')
plt.tight_layout(rect=[0,0.05,1,1])
p2 = os.path.join(OUTPUT_DIR,'plot2_diverging_likert.png')
plt.savefig(p2, dpi=150, bbox_inches='tight', facecolor=BG); plt.show()
print(f'✅  Saved: {p2}')

# ── Plot 3: Mann-Whitney Heatmap ──────────────────────────────────────────────
all_factors=['V_P','R_A','C_S','B_I','C_T','A_D','V_C']
short_names=['Purchase\nIntent','Range\nAnxiety','Charging\nInfrastr.',
             'Battery\nIssues','Charging\nTime','Autonomous\nDriving','V2X\nConnect.']
pmat=np.ones((len(all_factors),len(all_factors)))
for i,f1 in enumerate(all_factors):
    for j,f2 in enumerate(all_factors):
        if i!=j:
            _,p=stats.mannwhitneyu(df[f1],df[f2],alternative='two-sided')
            pmat[i,j]=p
fig, ax = plt.subplots(figsize=(10,8))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
norm=mcolors.LogNorm(vmin=1e-50,vmax=1.0)
im=ax.imshow(pmat,cmap=plt.cm.RdYlGn_r,norm=norm,aspect='auto')
for i in range(len(all_factors)):
    for j in range(len(all_factors)):
        p=pmat[i,j]
        if i==j:
            ax.text(j,i,'—',ha='center',va='center',fontsize=13,color='#888')
        else:
            if p<0.001: txt,sig,col=f'{p:.1e}','***','white'
            elif p<0.01: txt,sig,col=f'{p:.3f}','**','white'
            elif p<0.05: txt,sig,col=f'{p:.3f}','*','#222'
            else: txt,sig,col=f'{p:.3f}','n.s.','#222'
            ax.text(j,i,f'{txt}\n{sig}',ha='center',va='center',fontsize=7.5,color=col,fontweight='500')
ax.set_xticks(range(len(all_factors))); ax.set_yticks(range(len(all_factors)))
ax.set_xticklabels(short_names,fontsize=9); ax.set_yticklabels(short_names,fontsize=9)
ax.tick_params(length=0)
cbar=fig.colorbar(im,ax=ax,fraction=0.035,pad=0.03)
cbar.set_label('p-value (log scale)',fontsize=9); cbar.ax.tick_params(labelsize=8)
legend_elements=[
    mpatches.Patch(facecolor='#d73027',label='*** p < 0.001  (highly significant)'),
    mpatches.Patch(facecolor='#f4a582',label='**  p < 0.01'),
    mpatches.Patch(facecolor='#fee08b',label='*   p < 0.05'),
    mpatches.Patch(facecolor='#91cf60',label='n.s. p ≥ 0.05  (not significant)'),
]
ax.legend(handles=legend_elements,loc='upper left',bbox_to_anchor=(1.18,1.0),
          fontsize=8.5,title='Significance',framealpha=0.9)
ax.set_title('Mann-Whitney U Test — Pairwise P-value Matrix\n'
             'Are two perception dimensions rated differently by respondents?',
             fontsize=12,fontweight='bold',pad=14)
ax.text(0.5,-0.10,'Red = significantly different  |  Green = similar distributions\n'
        'Mann-Whitney U used because Likert data is ordinal (not normally distributed)',
        transform=ax.transAxes,ha='center',fontsize=8,color='#555',style='italic')
plt.tight_layout(rect=[0,0.05,0.88,1])
p3=os.path.join(OUTPUT_DIR,'plot3_mannwhitney_heatmap.png')
plt.savefig(p3,dpi=150,bbox_inches='tight',facecolor=BG); plt.show()
print(f'✅  Saved: {p3}')

# ── Plot 4: Dendrogram ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13,6))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
dendrogram(linked,ax=ax,color_threshold=8,above_threshold_color='#aaaaaa',
           no_labels=True,link_color_func=lambda k:ACCENT)
ax.axhline(y=8,color='#e74c3c',linewidth=1.5,linestyle='--',alpha=0.8,
           label='Cut-off line → defines number of clusters')
for xpos,label in {0.08:'Cluster A\n(EV-positive\nhigh interest)',
                   0.40:'Cluster B\n(Cautious /\nneutral)',
                   0.72:'Cluster C\n(Low interest\n/ sceptical)'}.items():
    ax.text(xpos,-3.2,label,transform=ax.get_xaxis_transform(),
            ha='center',fontsize=8.5,color='#333',
            bbox=dict(boxstyle='round,pad=0.3',facecolor='#eaf4fb',edgecolor=ACCENT,alpha=0.85))
ax.set_ylabel('Ward Linkage Distance\n(larger = more dissimilar)',fontsize=10,color='#444')
ax.set_xlabel('Individual Survey Respondents  (each leaf = 1 person)',fontsize=10,color='#444')
ax.yaxis.grid(True,color=GRID,linewidth=0.6); ax.set_axisbelow(True)
for s in ax.spines.values(): s.set_visible(False)
ax.legend(fontsize=9,loc='upper right',framealpha=0.9)
ax.set_title('Hierarchical Clustering Dendrogram — Consumer Segment Discovery\n'
             'Respondents grouped by similarity across all 6 EV perception factors',
             fontsize=12,fontweight='bold',pad=12)
ax.text(0.5,-0.16,'Method: Ward linkage on Likert responses (R_A, C_S, B_I, C_T, A_D, V_C)\n'
        'Each branch merger = two respondents/groups with similar EV perception profiles',
        transform=ax.transAxes,ha='center',fontsize=8,color='#666',style='italic')
plt.tight_layout(rect=[0,0.08,1,1])
p4=os.path.join(OUTPUT_DIR,'plot4_dendrogram.png')
plt.savefig(p4,dpi=150,bbox_inches='tight',facecolor=BG); plt.show()
print(f'✅  Saved: {p4}')

# ── Plot 5: K-Means on PCA ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,7))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
cluster_names={0:'Cluster A — EV Enthusiasts\n(High importance across all factors)',
               1:'Cluster B — Cautious Considerers\n(Moderate ratings, range anxiety dominant)',
               2:'Cluster C — Low Engagement\n(Low perceived importance overall)'}
for cid in [0,1,2]:
    mask=clusters==cid
    ax.scatter(X_pca[mask,0],X_pca[mask,1],c=colors[cid],marker=markers[cid],
               s=60,alpha=0.72,edgecolors='white',linewidth=0.5,label=cluster_names[cid])
for i,(cx,cy) in enumerate(centres):
    ax.scatter(cx,cy,c=colors[i],s=220,marker='*',edgecolors='#222',linewidth=1.2,zorder=5)
    ax.annotate(f'Centre {i+1}',(cx,cy),textcoords='offset points',xytext=(8,8),
                fontsize=8,color=colors[i],fontweight='bold')
ax.set_xlabel(f'Principal Component 1  ({var1:.1f}% variance explained)\n→ Overall EV perception score',
              fontsize=10,color='#444')
ax.set_ylabel(f'Principal Component 2  ({var2:.1f}% variance explained)\n→ Technology vs. Infrastructure split',
              fontsize=10,color='#444')
ax.xaxis.grid(True,color=GRID,linewidth=0.5); ax.yaxis.grid(True,color=GRID,linewidth=0.5)
ax.set_axisbelow(True)
for s in ax.spines.values(): s.set_visible(False)
ax.legend(loc='lower right',fontsize=8.5,framealpha=0.92,title='Consumer Segment',title_fontsize=9)
ax.set_title('K-Means Clustering (k=3) on PCA-Reduced EV Perception Space\n'
             'Each point = one respondent  |  Stars = cluster centroids',
             fontsize=12,fontweight='bold',pad=12)
ax.text(0.5,-0.15,f'PCA reduces 6 Likert dimensions to 2 axes  |  Total variance explained: {var1+var2:.1f}%',
        transform=ax.transAxes,ha='center',fontsize=8,color='#666',style='italic')
plt.tight_layout(rect=[0,0.07,1,1])
p5=os.path.join(OUTPUT_DIR,'plot5_kmeans_pca.png')
plt.savefig(p5,dpi=150,bbox_inches='tight',facecolor=BG); plt.show()
print(f'✅  Saved: {p5}')

print('\n' + '═'*60)
print(f'All 5 plots saved to: {OUTPUT_DIR}')
print('═'*60)
print('In Google Colab: click the folder icon (left sidebar)')
print(f'→ Open the plots/ folder → right-click each PNG → Download')